# 3D-QSAR starter: RDKit + alignment + RF
This notebook is designed to run in Google Colab. It loads a small sample dataset from GitHub, generates 3D conformers (ETKDG), aligns to a template, computes a small set of 3D descriptors, and fits a baseline RandomForestRegressor.

In [ ]:
%%capture
!pip -q install rdkit-pypi pandas numpy scikit-learn tqdm matplotlib

In [ ]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors3D

RAW_URL = "https://raw.githubusercontent.com/madariaga15/3D-QSAR-starter/main/data/sample_dataset.csv"

In [ ]:
df = pd.read_csv(RAW_URL)
df.head()

In [ ]:
def mol_from_smiles(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    # ETKDG 3D embed
    params = AllChem.ETKDGv3()
    params.randomSeed = 0xf00d
    if AllChem.EmbedMolecule(mol, params) != 0:
        return None
    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception:
        try:
            AllChem.UFFOptimizeMolecule(mol)
        except Exception:
            pass
    return mol

DESCR_FUNCS = {
    'RadiusOfGyration': Descriptors3D.RadiusOfGyration,
    'Asphericity': Descriptors3D.Asphericity,
    'Eccentricity': Descriptors3D.Eccentricity,
    'InertialShapeFactor': Descriptors3D.InertialShapeFactor,
    'NPR1': Descriptors3D.NPR1,
    'NPR2': Descriptors3D.NPR2,
}

In [ ]:
mols = []
rmsds = []
template = None
prb_conf_id = 0

for smi in tqdm(df['smiles'], desc='Building 3D molecules'):
    mol = mol_from_smiles(smi)
    if mol is None:
        rmsds.append(np.nan)
        mols.append(None)
        continue
    if template is None:
        template = mol
        rmsds.append(0.0)
    else:
        rmsd = AllChem.AlignMol(mol, template, prbId=prb_conf_id, refId=prb_conf_id)
        rmsds.append(float(rmsd))
    mols.append(mol)

df['rmsd_to_template'] = rmsds
df['mol'] = mols
df.head()

In [ ]:
X_rows = []
valid_targets = []
valid_idx = []

for i, (mol, target) in enumerate(zip(df['mol'], df['target'])):
    if mol is None or np.isnan(target):
        continue
    feats = {}
    for name, fn in DESCR_FUNCS.items():
        feats[name] = float(fn(mol))
    feats['rmsd_to_template'] = float(df.loc[i, 'rmsd_to_template'])
    X_rows.append(feats)
    valid_targets.append(float(target))
    valid_idx.append(i)

X = pd.DataFrame(X_rows)
y = pd.Series(valid_targets, name='target')
X.head(), y.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

rf = RandomForestRegressor(n_estimators=300, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print(f"R^2: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

## Next steps
- Replace the demo dataset with your own (SMILES + activity/value)
- Use a pharmacophore or atom-mapping alignment
- Expand descriptor set (e.g. rdMolDescriptors.CalcAutoCorr3D)
- Tune model hyperparameters and validation